In [17]:
import numpy as np
import pandas as pd
from fractions import Fraction
from math import floor, gcd, log, ceil
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate, QFT
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
import time

def build_mod_mult_unitary(k, N, n_bits):
    """Build the unitary matrix for 'multiply by k mod N'.
    
    Each input state |x> maps to |x*k mod N> for valid inputs.
    This is a permutation matrix — always unitary.
    """
    dim = 2**n_bits
    U = np.zeros((dim, dim))
    for x in range(dim):
        if x == 0:
            U[0][0] = 1                      # 0 maps to 0
        elif x < N and gcd(x, N) == 1:
            y = (x * k) % N                  # Valid input: multiply mod N
            U[y][x] = 1
        else:
            U[x][x] = 1                      # Out-of-range: identity
    return U

def build_shor_circuit(N, a, num_control, num_target):
    """Build the complete Shor's circuit with named registers.
    
    Returns the circuit and the number of controlled multiplications applied.
    """
    # Named registers for clear circuit diagrams
    control = QuantumRegister(num_control, name="Counting")
    target = QuantumRegister(num_target, name="Target")
    output = ClassicalRegister(num_control, name="output")
    circuit = QuantumCircuit(control, target, output)
    
    # Step 1: Initialise target register to |1>
    circuit.x(num_control)  # Set first target qubit to |1>
    
    # Step 2: Hadamard on all counting qubits (creates superposition)
    for k in range(num_control):
        circuit.h(k)
    
    circuit.barrier()
    
    # Step 3: Controlled modular exponentiation
    n_mod_mults = 0
    for k in range(num_control):
        # Compute what constant this qubit multiplies by
        power = pow(a, 2**k, N)
        
        if power == 1:
            continue  # Multiplying by 1 does nothing, skip
        
        # Build unitary gate for this multiplication
        U = build_mod_mult_unitary(power, N, num_target)
        gate = UnitaryGate(U)
        gate.name = f"M_{power}"
        
        # Make it controlled by counting qubit k
        c_gate = gate.control()
        circuit.compose(c_gate, qubits=[k] + list(range(num_control, num_control + num_target)), inplace=True)
        n_mod_mults += 1
    
    circuit.barrier()
    
    # Step 4: Inverse QFT on counting register
    circuit.compose(QFT(num_control, inverse=True), qubits=control, inplace=True)
    
    circuit.barrier()
    
    # Step 5: Measure counting register
    circuit.measure(control, output)
    
    return circuit, n_mod_mults

In [ ]:
# For N=15, we need gates for "multiply by 2" and "multiply by 4"
# (other powers of a=2 reduce to 1 mod 15)
N = 15
num_target = ceil(log(N, 2))  # 4 qubits

print(f"Modular multiplication gates for N={N}:")
print(f"Target register: {num_target} qubits\n")

for k_val in [2, 4]:
    U = build_mod_mult_unitary(k_val, N, num_target)
    gate = UnitaryGate(U)
    gate.name = f"M_{k_val}"
    
    # Show the gate in a small circuit
    circ = QuantumCircuit(num_target)
    circ.append(gate, range(num_target))
    print(f"--- M_{k_val} (multiply by {k_val} mod {N}) ---")
    print(circ.draw(output='text'))
    print()

In [ ]:
N = 15
a = 2
num_target = ceil(log(N, 2))         # 4 qubits for target register
num_control = 2 * num_target          # 8 qubits for counting register

print(f"Shor's Algorithm Circuit: N={N}, a={a}")
print(f"Counting qubits: {num_control}, Target qubits: {num_target}")
print(f"Total qubits: {num_control + num_target}\n")

# Show which modular multiplications each counting qubit controls
print("Counting qubit → Controlled multiplication:")
for k in range(num_control):
    power = pow(a, 2**k, N)
    skip = " (skip: identity)" if power == 1 else ""
    print(f"  Qubit {k}: a^(2^{k}) mod {N} = {a}^{2**k} mod {N} = {power}{skip}")

# Build the circuit
circuit_15, n_mults_15 = build_shor_circuit(N, a, num_control, num_target)

print(f"\nControlled multiplications applied: {n_mults_15}")
print(f"\nCircuit diagram:")
circuit_15.draw(output='mpl', fold=-1)

In [ ]:
backend = AerSimulator()
compiled_15 = transpile(circuit_15, backend, optimization_level=1)

print(f"Transpiled circuit:")
print(f"  Total gates: {sum(compiled_15.count_ops().values())}")
print(f"  Gate breakdown: {dict(compiled_15.count_ops())}")
print(f"  Circuit depth: {compiled_15.depth()}")

# Run simulation
shots = 4096
start = time.perf_counter()
result = backend.run(compiled_15, shots=shots).result()
sim_time = (time.perf_counter() - start) * 1000

counts_15 = result.get_counts()
print(f"\n  Simulation time: {sim_time:.1f} ms ({shots} shots)")
print(f"  Time per shot: {sim_time/shots:.4f} ms")

# Show histogram of results
plot_histogram(counts_15, figsize=(12, 4), title=f"Shor's Algorithm Results: N={N}, a={a}")


In [ ]:
rows_phase = []
measured_phases = []

for bitstring in sorted(counts_15, key=lambda x: counts_15[x], reverse=True):
    count = counts_15[bitstring]
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control)
    measured_phases.append(phase)
    rows_phase.append([
        bitstring,
        decimal,
        count,
        f"{count/shots:.1%}",
        f"{decimal}/{2**num_control} = {phase:.4f}"
    ])

print(f"Phase extraction for N={N}, a={a}:")
df_phase = pd.DataFrame(rows_phase[:15],
    columns=["Bitstring", "Decimal", "Count", "Probability", "Phase"])
print(df_phase.to_string(index=False))

In [ ]:
rows_period = []
success_shots = 0

for bitstring in sorted(counts_15, key=lambda x: counts_15[x], reverse=True):
    count = counts_15[bitstring]
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control)
    
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator
    
    # Validate period
    valid = r > 1 and pow(a, r, N) == 1
    
    # Try to extract factors
    factors = ""
    if valid and r % 2 == 0:
        x = pow(a, r // 2, N)
        if x != N - 1:
            f1, f2 = gcd(x - 1, N), gcd(x + 1, N)
            if 1 < f1 < N:
                factors = f"{f1} × {N // f1}"
                success_shots += count
            elif 1 < f2 < N:
                factors = f"{f2} × {N // f2}"
                success_shots += count
    
    rows_period.append([
        f"{phase:.4f}",
        f"{frac.numerator}/{frac.denominator}",
        r,
        "Yes" if valid else "No",
        count,
        factors if factors else "—"
    ])

print(f"Period extraction and factoring for N={N}, a={a}:")
df_period = pd.DataFrame(rows_period[:15],
    columns=["Phase", "Fraction", "Guess for r", "Valid", "Count", "Factors"])
print(df_period.to_string(index=False))

print(f"\nSuccess rate: {100*success_shots/shots:.1f}% ({success_shots}/{shots} shots)")
if success_shots > 0:
    print(f"Avg shots to first success: {round(shots/success_shots)}")

In [ ]:
print("=" * 90)
print(f"SHOR'S ALGORITHM: N = 15 — ALL COPRIME BASES")
print("=" * 90)

N = 15
num_target = ceil(log(N, 2))
num_control = 2 * num_target
shots = 4096
bases_15 = [a for a in range(2, N) if gcd(a, N) == 1]

print(f"\nCoprime bases: {bases_15}")
print(f"Circuit: {num_control} counting + {num_target} target = {num_control + num_target} qubits, {shots} shots\n")

results_15 = []

for a in bases_15:
    # True period (for reference)
    val, true_r = a % N, 1
    while val != 1:
        val = (val * a) % N
        true_r += 1
    
    # Build and run
    circuit, n_mults = build_shor_circuit(N, a, num_control, num_target)
    compiled = transpile(circuit, AerSimulator(), optimization_level=1)
    
    start = time.perf_counter()
    result = AerSimulator().run(compiled, shots=shots).result()
    sim_time = (time.perf_counter() - start) * 1000
    
    counts = result.get_counts()
    gate_counts = compiled.count_ops()
    
    # Extract success rate
    success = 0
    factors_found = set()
    for bitstring, count in counts.items():
        phase = int(bitstring, 2) / (2**num_control)
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        if r <= 1 or pow(a, r, N) != 1 or r % 2 != 0:
            continue
        x = pow(a, r // 2, N)
        if x == N - 1:
            continue
        f1, f2 = gcd(x - 1, N), gcd(x + 1, N)
        if 1 < f1 < N:
            factors_found.add((min(f1, N//f1), max(f1, N//f1)))
            success += count
        elif 1 < f2 < N:
            factors_found.add((min(f2, N//f2), max(f2, N//f2)))
            success += count
    
    sr = 100 * success / shots
    s2s = round(1 / (success/shots)) if success > 0 else None
    
    results_15.append({
        'a': a, 'true_r': true_r, 'n_mults': n_mults,
        'total_gates': sum(gate_counts.values()),
        'depth': compiled.depth(),
        'sim_time': sim_time,
        'success_rate': sr,
        'shots_to_1st': s2s,
        'factors': list(factors_found)
    })

# Display as table
rows = []
for r in results_15:
    f_str = str(r['factors'][0]) if r['factors'] else "—"
    s2s_str = str(r['shots_to_1st']) if r['shots_to_1st'] else "—"
    rows.append([r['a'], r['true_r'], r['n_mults'], r['total_gates'],
                 r['depth'], f"{r['sim_time']:.1f}", f"{r['success_rate']:.1f}%",
                 s2s_str, f_str])

df_15 = pd.DataFrame(rows,
    columns=["Base a", "Period r", "Ctrl Mults", "Total Gates", "Depth",
             "Sim Time (ms)", "Success Rate", "Shots to 1st", "Factors"])
print(df_15.to_string(index=False))

successful = [r for r in results_15 if r['factors']]
print(f"\nBases that found factors: {[r['a'] for r in successful]} ({len(successful)}/{len(bases_15)})")
if successful:
    print(f"Avg success rate: {np.mean([r['success_rate'] for r in successful]):.1f}%")

In [ ]:
N = 21
a = 2
num_target = ceil(log(N, 2))         # 5 qubits
num_control = 2 * num_target          # 10 qubits

print(f"Shor's Algorithm Circuit: N={N}, a={a}")
print(f"Counting qubits: {num_control}, Target qubits: {num_target}")
print(f"Total qubits: {num_control + num_target}\n")

print("Counting qubit → Controlled multiplication:")
for k in range(num_control):
    power = pow(a, 2**k, N)
    skip = " (skip: identity)" if power == 1 else ""
    print(f"  Qubit {k}: a^(2^{k}) mod {N} = {a}^{2**k} mod {N} = {power}{skip}")

circuit_21, n_mults_21 = build_shor_circuit(N, a, num_control, num_target)

print(f"\nControlled multiplications applied: {n_mults_21}")
print(f"\nCircuit diagram:")
circuit_21.draw(output='mpl', fold=-1)

In [ ]:
backend = AerSimulator()
compiled_21 = transpile(circuit_21, backend, optimization_level=1)

print(f"Transpiled circuit:")
print(f"  Total gates: {sum(compiled_21.count_ops().values())}")
print(f"  Gate breakdown: {dict(compiled_21.count_ops())}")
print(f"  Circuit depth: {compiled_21.depth()}")

shots = 4096
start = time.perf_counter()
result = backend.run(compiled_21, shots=shots).result()
sim_time = (time.perf_counter() - start) * 1000

counts_21 = result.get_counts()
print(f"\n  Simulation time: {sim_time:.1f} ms ({shots} shots)")
print(f"  Time per shot: {sim_time/shots:.4f} ms")

plot_histogram(counts_21, figsize=(12, 4), title=f"Shor's Algorithm Results: N={N}, a={a}")

In [ ]:
N, a = 21, 2
num_control_21 = 2 * ceil(log(N, 2))

# Phase table
rows_phase = []
for bitstring in sorted(counts_21, key=lambda x: counts_21[x], reverse=True):
    count = counts_21[bitstring]
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control_21)
    rows_phase.append([bitstring, decimal, count, f"{count/shots:.1%}",
                       f"{decimal}/{2**num_control_21} = {phase:.4f}"])

print(f"Phase extraction for N={N}, a={a}:")
df = pd.DataFrame(rows_phase[:15],
    columns=["Bitstring", "Decimal", "Count", "Probability", "Phase"])
print(df.to_string(index=False))

# Period and factor table
rows_period = []
success_shots_21 = 0

for bitstring in sorted(counts_21, key=lambda x: counts_21[x], reverse=True):
    count = counts_21[bitstring]
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control_21)
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator
    valid = r > 1 and pow(a, r, N) == 1
    
    factors = ""
    if valid and r % 2 == 0:
        x = pow(a, r // 2, N)
        if x != N - 1:
            f1, f2 = gcd(x - 1, N), gcd(x + 1, N)
            if 1 < f1 < N:
                factors = f"{f1} × {N // f1}"
                success_shots_21 += count
            elif 1 < f2 < N:
                factors = f"{f2} × {N // f2}"
                success_shots_21 += count
    
    rows_period.append([f"{phase:.4f}", f"{frac.numerator}/{frac.denominator}",
                        r, "Yes" if valid else "No", count, factors if factors else "—"])

print(f"\nPeriod extraction and factoring for N={N}, a={a}:")
df = pd.DataFrame(rows_period[:15],
    columns=["Phase", "Fraction", "Guess for r", "Valid", "Count", "Factors"])
print(df.to_string(index=False))

print(f"\nSuccess rate: {100*success_shots_21/shots:.1f}% ({success_shots_21}/{shots} shots)")
if success_shots_21 > 0:
    print(f"Avg shots to first success: {round(shots/success_shots_21)}")


In [ ]:
print("=" * 90)
print(f"SHOR'S ALGORITHM: N = 21 — ALL COPRIME BASES")
print("=" * 90)

N = 21
num_target = ceil(log(N, 2))
num_control = 2 * num_target
shots = 4096
bases_21 = [a for a in range(2, N) if gcd(a, N) == 1]

print(f"\nCoprime bases: {bases_21}")
print(f"Circuit: {num_control} counting + {num_target} target = {num_control + num_target} qubits, {shots} shots\n")

results_21 = []

for a in bases_21:
    val, true_r = a % N, 1
    while val != 1:
        val = (val * a) % N
        true_r += 1
    
    circuit, n_mults = build_shor_circuit(N, a, num_control, num_target)
    compiled = transpile(circuit, AerSimulator(), optimization_level=1)
    
    start = time.perf_counter()
    result = AerSimulator().run(compiled, shots=shots).result()
    sim_time = (time.perf_counter() - start) * 1000
    
    counts = result.get_counts()
    gate_counts = compiled.count_ops()
    
    success = 0
    factors_found = set()
    for bitstring, count in counts.items():
        phase = int(bitstring, 2) / (2**num_control)
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        if r <= 1 or pow(a, r, N) != 1 or r % 2 != 0:
            continue
        x = pow(a, r // 2, N)
        if x == N - 1:
            continue
        f1, f2 = gcd(x - 1, N), gcd(x + 1, N)
        if 1 < f1 < N:
            factors_found.add((min(f1, N//f1), max(f1, N//f1)))
            success += count
        elif 1 < f2 < N:
            factors_found.add((min(f2, N//f2), max(f2, N//f2)))
            success += count
    
    sr = 100 * success / shots
    s2s = round(1 / (success/shots)) if success > 0 else None
    
    results_21.append({
        'a': a, 'true_r': true_r, 'n_mults': n_mults,
        'total_gates': sum(gate_counts.values()),
        'depth': compiled.depth(),
        'sim_time': sim_time,
        'success_rate': sr,
        'shots_to_1st': s2s,
        'factors': list(factors_found)
    })

rows = []
for r in results_21:
    f_str = str(r['factors'][0]) if r['factors'] else "—"
    s2s_str = str(r['shots_to_1st']) if r['shots_to_1st'] else "—"
    rows.append([r['a'], r['true_r'], r['n_mults'], r['total_gates'],
                 r['depth'], f"{r['sim_time']:.1f}", f"{r['success_rate']:.1f}%",
                 s2s_str, f_str])

df_21 = pd.DataFrame(rows,
    columns=["Base a", "Period r", "Ctrl Mults", "Total Gates", "Depth",
             "Sim Time (ms)", "Success Rate", "Shots to 1st", "Factors"])
print(df_21.to_string(index=False))

successful = [r for r in results_21 if r['factors']]
print(f"\nBases that found factors: {[r['a'] for r in successful]} ({len(successful)}/{len(bases_21)})")
if successful:
    print(f"Avg success rate: {np.mean([r['success_rate'] for r in successful]):.1f}%")

In [ ]:
print("=" * 90)
print("QUANTUM SCALABILITY SUMMARY: N=15 vs N=21")
print("=" * 90)

for label, stored in [("N=15", results_15), ("N=21", results_21)]:
    successful = [r for r in stored if r['factors']]
    failed = [r for r in stored if not r['factors']]
    
    print(f"\n--- {label} ---")
    print(f"  Coprime bases tested:        {len(stored)}")
    print(f"  Bases that found factors:    {len(successful)} ({[r['a'] for r in successful]})")
    print(f"  Bases that failed:           {len(failed)} ({[r['a'] for r in failed]})")
    
    if successful:
        rows_summary = []
        rows_summary.append(["Success rate", f"{np.mean([r['success_rate'] for r in successful]):.1f}%"])
        rows_summary.append(["Avg shots to 1st success", f"{np.mean([r['shots_to_1st'] for r in successful if r['shots_to_1st']]):.1f}"])
        rows_summary.append(["Avg period r", f"{np.mean([r['true_r'] for r in successful]):.1f}"])
        rows_summary.append(["Avg ctrl multiplications", f"{np.mean([r['n_mults'] for r in successful]):.1f}"])
        rows_summary.append(["Avg total gates", f"{np.mean([r['total_gates'] for r in successful]):.0f}"])
        rows_summary.append(["Avg circuit depth", f"{np.mean([r['depth'] for r in successful]):.0f}"])
        rows_summary.append(["Avg simulation time", f"{np.mean([r['sim_time'] for r in successful]):.1f} ms"])
        
        df_summary = pd.DataFrame(rows_summary, columns=["Metric", "Value"])
        print(f"\n  Averages (successful bases):")
        print(df_summary.to_string(index=False))

# Growth comparison
s15 = [r for r in results_15 if r['factors']]
s21 = [r for r in results_21 if r['factors']]
if s15 and s21:
    print(f"\n--- Growth from N=15 to N=21 ---")
    rows_growth = []
    g15 = np.mean([r['total_gates'] for r in s15])
    g21 = np.mean([r['total_gates'] for r in s21])
    d15 = np.mean([r['depth'] for r in s15])
    d21 = np.mean([r['depth'] for r in s21])
    t15 = np.mean([r['sim_time'] for r in s15])
    t21 = np.mean([r['sim_time'] for r in s21])
    
    rows_growth.append(["Gate count", f"{g15:.0f}", f"{g21:.0f}", f"{g21/g15:.1f}x"])
    rows_growth.append(["Circuit depth", f"{d15:.0f}", f"{d21:.0f}", f"{d21/d15:.1f}x"])
    rows_growth.append(["Simulation time (ms)", f"{t15:.1f}", f"{t21:.1f}", f"{t21/t15:.1f}x"])
    
    df_growth = pd.DataFrame(rows_growth, columns=["Metric", "N=15", "N=21", "Growth"])
    print(df_growth.to_string(index=False))